# 🧹 Limpeza e Padronização — Camada Silver (SISCAN / 10ª CRS)

Este notebook transforma os dados brutos da **Camada Bronze** em dados limpos, padronizados e estruturados na **Camada Silver** da Arquitetura Medallion.

---

## Limpezas aplicadas:

1. **Separação código/nome do município** — coluna `Munic.de residencia` (ex: `430040 ALEGRETE`) é dividida em `codigo_municipio` (int) e `municipio` (string), padronizada com `initcap` e remoção de barras invertidas
2. **Filtro estrito da 10ª CRS** — apenas os 11 municípios oficiais (lista `CODIGOS_10CRS`) são mantidos; linhas `Total` e municípios fora da CRS são removidos
3. **Correção de encoding** — nomes de colunas com caracteres corrompidos (ex: `MAR�O` → mês 3) são normalizados via mapeamento centralizado (`MONTH_MAP` / `MONTH_EXPR`)
4. **Cast de tipos sem "zero forçado"** — `quantidade_raw` (string original) é convertida para `quantidade` (int) com `try_cast` sem `coalesce(…, 0)`, preservando NULL onde o dado era ausente
5. **Unpivot (wide → long)** — formato matricial (meses/faixas etárias como colunas) é convertido para formato tidy (uma linha por observação)
6. **Extração de metadados do arquivo** — `ano`, `categoria_adequabilidade` e `grupo_resultado` são extraídos do nome do arquivo `_file_source`
7. **Coluna `dt_competencia`** — data no primeiro dia do mês (tipo DATE) para tabelas com granularidade mensal (`exames_geral` e `adequabilidade`)
8. **Coluna `fl_populacao_alvo`** — flag booleana (25-64 anos, diretrizes MS) identificada via regex robusta (`REGEX_POP_ALVO`) contra espaços extras do TABNET
9. **Coluna `grupo_resultado`** — explicita o filtro da extração DATASUS na tabela `resultados_citopatologicos`
10. **Preservação de origem** — colunas `_file_source`, `_ingestion_time`, `_bronze_source` e `_silver_processed_time` garantem rastreabilidade completa
11. **Validação estrita pré-gravação** — PKs, nulos em colunas obrigatórias e duplicidades são validados **antes** do `.write.saveAsTable()`

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, FloatType, IntegerType

# Catálogo e schemas
CATALOG = "`tcc-analise-siscan-10crs`"
SCHEMA_BRONZE = "bronze"
SCHEMA_SILVER = "silver"

# 1. Escopo Temporal
ANO_INICIO = 2020
ANO_FIM = 2025

# 2. Dicionário Oficial de Municípios da 10ª CRS (11 Municípios)
MUNICIPIOS_10CRS = {
    430040: "Alegrete",
    430187: "Barra do Quarai",
    431060: "Itaqui",
    431171: "Macambara",
    431175: "Manoel Viana",
    431530: "Quarai",
    431640: "Rosario do Sul",
    431697: "Santa Margarida do Sul",
    431710: "Sant'Ana do Livramento",
    431830: "Sao Gabriel",
    432240: "Uruguaiana"
}
CODIGOS_10CRS = list(MUNICIPIOS_10CRS.keys())

# 3. Mapeamento Único de Meses (Tratamento de Encoding e Acentos)
MONTH_MAP = {
    "JANEIRO": 1, "FEVEREIRO": 2, "MARCO": 3, "MARO": 3, "MARÇO": 3,
    "ABRIL": 4, "MAIO": 5, "JUNHO": 6, "JULHO": 7, "AGOSTO": 8,
    "SETEMBRO": 9, "OUTUBRO": 10, "NOVEMBRO": 11, "DEZEMBRO": 12
}

# Expressão SQL centralizada para converter mês para número (strip non-alpha p/ encoding corrompido)
MONTH_EXPR = "CASE " + " ".join([f"WHEN UPPER(REGEXP_REPLACE(SUBSTR(mes_ano_col, 1, INSTR(mes_ano_col, '/') - 1), '[^A-Za-z]', '')) = '{k}' THEN {v}" for k, v in MONTH_MAP.items()]) + " ELSE NULL END"

# 4. Faixas Etárias da População-Alvo (25-64 anos) — Diretrizes do Ministério da Saúde
FAIXAS_ALVO = [
    "25 a 29 anos", "30 a 34 anos", "35 a 39 anos", "40 a 44 anos",
    "45 a 49 anos", "50 a 54 anos", "55 a 59 anos", "60 a 64 anos"
]

# Regex para identificar população-alvo (25-64 anos) — robusto contra espaços extras do TABNET
REGEX_POP_ALVO = r'^(2[5-9]|3[0-9]|4[0-9]|5[0-9]|6[0-4])\s+a\s+\d+\s+anos'

# Garantir Schema Silver
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_SILVER}")
print(f"✅ Schema {CATALOG}.{SCHEMA_SILVER} garantido com {len(CODIGOS_10CRS)} municípios na 10ª CRS.")

## 1. Limpeza: `exames_geral_raw` → `silver.exames_geral`

**Origem:** 1 arquivo CSV com todos os anos (2020-2025), formato wide (72 meses como colunas).

**Transformações:**
* Separar `codigo_municipio` e `municipio` da coluna `Munic.de residencia`
* Remover linha `Total`
* Unpivot dos 72 meses para formato long (`ano`, `mes`, `quantidade`)
* Remover coluna ` Total` agregada
* Padronizar nomes dos meses

In [0]:
# ---------------------------------------------------------------------------
# 1. SILVER: exames_geral (Q1 + Q2) — Unpivot meses para formato long
# ---------------------------------------------------------------------------

df_bronze = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.exames_geral_raw")

# 1a. Extração e Validação do Município
df_silver = (
    df_bronze
    .withColumn("codigo_municipio", F.expr("try_cast(substring(trim(`Munic.de residencia`), 1, 6) as int)"))
    .withColumn("municipio_raw", F.trim(F.substring(F.col("`Munic.de residencia`"), 7, 1000)))
    .filter(F.col("codigo_municipio").isin(CODIGOS_10CRS))
    .withColumn("municipio", F.initcap(F.regexp_replace(F.col("municipio_raw"), r"\\", "")))
)

# 1b. Identificar colunas mensais válidas
month_cols = [c for c in df_bronze.columns if "/" in c and c != "Munic.de residencia"]

# 1c. Unpivot mantendo a fonte do arquivo e ingestion_time
df_long = df_silver.unpivot(
    ids=["codigo_municipio", "municipio", "_file_source", "_ingestion_time"],
    values=month_cols,
    variableColumnName="mes_ano_col",
    valueColumnName="quantidade_raw"
)

# 1d. Tratamento do Período e Quantidade sem "Zero Falso"
df_exames_geral = (
    df_long
    .withColumn("ano", F.expr("try_cast(regexp_extract(mes_ano_col, '/([0-9]{4})$', 1) as int)"))
    .withColumn("mes", F.expr(f"try_cast({MONTH_EXPR} as int)"))
    .withColumn("dt_competencia", F.expr("try_to_date(concat_ws('-', ano, lpad(mes, 2, '0'), '01'), 'yyyy-MM-dd')"))
    .withColumn("quantidade", F.expr("try_cast(quantidade_raw as int)"))
    .withColumn("_bronze_source", F.lit(f"{CATALOG}.{SCHEMA_BRONZE}.exames_geral_raw"))
    .withColumn("_silver_processed_time", F.current_timestamp())
    .select(
        "codigo_municipio", "municipio", "ano", "mes", "dt_competencia",
        "quantidade_raw", "quantidade", "_file_source", "_ingestion_time",
        "_bronze_source", "_silver_processed_time"
    )
    .orderBy("codigo_municipio", "dt_competencia")
)

print(f"exames_geral: {df_exames_geral.count()} linhas preparadas.")
display(df_exames_geral.limit(10))

## 2. Limpeza: `faixa_etaria_raw` → `silver.faixa_etaria`

**Origem:** 6 arquivos CSV (1 por ano 2020-2025), formato wide (faixas etárias como colunas, todas como STRING).

**Transformações:**
* Extrair `ano` do nome do arquivo `_file_source`
* Separar `codigo_municipio` e `municipio`
* Remover linha `Total`
* Cast de STRING → INTEGER em todas as colunas de faixa etária
* Unpivot das faixas etárias para formato long (`faixa_etaria`, `quantidade`)
* Nulos substituídos por 0

In [0]:
# ---------------------------------------------------------------------------
# 2. SILVER: faixa_etaria (Q3) — Extrair ano do arquivo + unpivot faixas
# ---------------------------------------------------------------------------

df_bronze = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.faixa_etaria_raw")

# 2a. Extrair ano do nome do arquivo (ex: ..._10CRS_2020.csv -> 2020)
df_silver = (
    df_bronze
    .withColumn("ano", F.expr("try_cast(regexp_extract(_file_source, '([0-9]{4})\\.csv$', 1) as int)"))
    .withColumn("codigo_municipio", F.expr("try_cast(substring(trim(`Munic.de residencia`), 1, 6) as int)"))
    .withColumn("municipio", F.initcap(F.regexp_replace(F.trim(F.substring(F.col("`Munic.de residencia`"), 7, 1000)), r"\\", "")))
    .filter(F.col("codigo_municipio").isin(CODIGOS_10CRS))
)

# 2b. Identificar colunas de faixa etária dinamicamente
exclude_cols = {"Munic.de residencia", "_ingestion_time", "_file_source", " Total", "ano", "codigo_municipio", "municipio"}
faixa_cols = [c for c in df_silver.columns if c not in exclude_cols]

# 2c. Unpivot mantendo quantidade_raw em formato String antes da conversão
df_long = df_silver.unpivot(
    ids=["codigo_municipio", "municipio", "ano", "_file_source", "_ingestion_time"],
    values=faixa_cols,
    variableColumnName="faixa_etaria_raw",
    valueColumnName="quantidade_raw"
)

# 2d. Normalizar faixas etárias + flag populacao-alvo (regex robusto) + quantidade sem zero forçado
df_faixa_etaria = (
    df_long
    .withColumn("faixa_etaria", F.regexp_replace(F.trim(F.col("faixa_etaria_raw")), r"^Entre\s+", ""))
    .withColumn("fl_populacao_alvo", F.col("faixa_etaria").rlike(REGEX_POP_ALVO))
    .withColumn("quantidade", F.expr("try_cast(quantidade_raw as int)"))
    .withColumn("_bronze_source", F.lit(f"{CATALOG}.{SCHEMA_BRONZE}.faixa_etaria_raw"))
    .withColumn("_silver_processed_time", F.current_timestamp())
    .select(
        "codigo_municipio", "municipio", "ano", "faixa_etaria_raw", "faixa_etaria",
        "fl_populacao_alvo", "quantidade_raw", "quantidade", "_file_source",
        "_ingestion_time", "_bronze_source", "_silver_processed_time"
    )
    .orderBy("codigo_municipio", "ano", "faixa_etaria")
)

print(f"faixa_etaria: {df_faixa_etaria.count()} linhas preparadas.")
display(df_faixa_etaria.limit(10))

## 3. Limpeza: `adequabilidade_raw` → `silver.adequabilidade`

**Origem:** 3 arquivos CSV (1 por categoria: Satisfatória, Insatisfatória, Rejeitada), formato wide (72 meses como colunas, tipos mistos STRING/INT).

**Transformações:**
* Extrair `categoria_adequabilidade` do nome do arquivo `_file_source`
* Separar `codigo_municipio` e `municipio`
* Remover linhas `Total`
* Cast de STRING → INTEGER em todas as colunas de meses (tipos mistos)
* Unpivot dos 72 meses para formato long (`ano`, `mes`, `quantidade`)
* Nulos substituídos por 0

In [0]:
# ---------------------------------------------------------------------------
# 3. SILVER: adequabilidade (Q4) — Extrair categoria do arquivo + unpivot meses
# ---------------------------------------------------------------------------

df_bronze = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.adequabilidade_raw")

# 3a. Extrair categoria de adequabilidade do nome do arquivo
# Ex: 'satisfatório%20-cito...' -> 'Satisfatória'; 'rejeitada%20-cito...' -> 'Rejeitada'
df_silver = (
    df_bronze
    .withColumn("categoria_adequabilidade",
        F.when(F.lower(F.col("_file_source")).contains("insatisfat"), F.lit("Insatisfatoria"))
         .when(F.lower(F.col("_file_source")).contains("satisfat"), F.lit("Satisfatoria"))
         .when(F.lower(F.col("_file_source")).contains("rejeitada"), F.lit("Rejeitada"))
         .otherwise(F.lit("Desconhecida"))
    )
)

# 3b. Separar código e nome do município (initcap + remove barras invertidas)
df_silver = (
    df_silver
    .withColumn("codigo_municipio", F.expr("try_cast(substring(trim(`Munic.de residencia`), 1, 6) as int)"))
    .withColumn("municipio", F.initcap(F.regexp_replace(F.trim(F.substring(F.col("`Munic.de residencia`"), 7, 1000)), r"\\", "")))
    .filter(F.col("codigo_municipio").isin(CODIGOS_10CRS))
)

# 3c. Identificar colunas de meses dinamicamente
month_cols = [c for c in df_bronze.columns if "/" in c and c != "Munic.de residencia"]

# 3c.1 Cast de todas as colunas de meses para STRING (tipos mistos INT/STRING no bronze)
cast_exprs = {c: F.col(f"`{c}`").cast("string") for c in month_cols}
df_silver = df_silver.withColumns(cast_exprs)

# 3d. Unpivot mantendo origem e quantidade_raw
df_long = df_silver.unpivot(
    ids=["codigo_municipio", "municipio", "categoria_adequabilidade", "_file_source", "_ingestion_time"],
    values=month_cols,
    variableColumnName="mes_ano_col",
    valueColumnName="quantidade_raw"
)

# 3e. Tratamento do Período e Quantidade sem "Zero Falso"
df_adequabilidade = (
    df_long
    .withColumn("ano", F.expr("try_cast(regexp_extract(mes_ano_col, '/([0-9]{4})$', 1) as int)"))
    .withColumn("mes", F.expr(f"try_cast({MONTH_EXPR} as int)"))
    .withColumn("dt_competencia", F.expr("try_to_date(concat_ws('-', ano, lpad(mes, 2, '0'), '01'), 'yyyy-MM-dd')"))
    .withColumn("quantidade", F.expr("try_cast(quantidade_raw as int)"))
    .withColumn("_bronze_source", F.lit(f"{CATALOG}.{SCHEMA_BRONZE}.adequabilidade_raw"))
    .withColumn("_silver_processed_time", F.current_timestamp())
    .select(
        "codigo_municipio", "municipio", "categoria_adequabilidade", "ano", "mes",
        "dt_competencia", "quantidade_raw", "quantidade", "_file_source",
        "_ingestion_time", "_bronze_source", "_silver_processed_time"
    )
    .orderBy("codigo_municipio", "categoria_adequabilidade", "dt_competencia")
)

print(f"adequabilidade: {df_adequabilidade.count()} linhas preparadas.")
display(df_adequabilidade.limit(10))

## 4. Limpeza: `resultados_raw` → `silver.resultados_citopatologicos`

**Origem:** 6 arquivos CSV (1 por ano 2020-2025), formato wide (faixas etárias como colunas STRING).

**Transformações:**
* Extrair `ano` e `grupo_resultado` do nome do arquivo `_file_source`
* Separar `codigo_municipio` e `municipio`
* Remover linhas `Total`
* Unpivot das faixas etárias para formato long (`faixa_etaria`, `quantidade_raw`)
* `quantidade` preserva NULL (try_cast sem coalesce 0)
* Flag `fl_populacao_alvo` via regex robusta (25-64 anos)
* Preservar `_file_source` e `_ingestion_time` para rastreabilidade

In [0]:
# ---------------------------------------------------------------------------
# 4. SILVER: resultados_citopatologicos (Q5) — Extrair ano + unpivot faixas
# ---------------------------------------------------------------------------

df_bronze = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.resultados_raw")

# 4a. Extrair ano e grupo_resultado do nome do arquivo
df_silver = (
    df_bronze
    .withColumn("ano", F.expr("try_cast(regexp_extract(_file_source, '([0-9]{4})\\.csv$', 1) as int)"))
    .withColumn("grupo_resultado", F.regexp_extract(F.col("_file_source"), r"([^/]+?)_[0-9]{4}\.csv", 1))
    .withColumn("codigo_municipio", F.expr("try_cast(substring(trim(`Munic.de residencia`), 1, 6) as int)"))
    .withColumn("municipio", F.initcap(F.regexp_replace(F.trim(F.substring(F.col("`Munic.de residencia`"), 7, 1000)), r"\\", "")))
    .filter(F.col("codigo_municipio").isin(CODIGOS_10CRS))
)

# 4b. Identificar colunas de faixa etária dinamicamente
exclude_cols = {"Munic.de residencia", "_ingestion_time", "_file_source", " Total", "ano", "codigo_municipio", "municipio", "grupo_resultado"}
faixa_cols = [c for c in df_silver.columns if c not in exclude_cols]

# 4c. Unpivot mantendo origem e quantidade_raw
df_long = df_silver.unpivot(
    ids=["codigo_municipio", "municipio", "ano", "grupo_resultado", "_file_source", "_ingestion_time"],
    values=faixa_cols,
    variableColumnName="faixa_etaria_raw",
    valueColumnName="quantidade_raw"
)

# 4d. Normalizar faixas etárias + flag populacao-alvo + quantidade sem zero forçado
df_resultados_citopatologicos = (
    df_long
    .withColumn("faixa_etaria", F.regexp_replace(F.trim(F.col("faixa_etaria_raw")), r"^Entre\s+", ""))
    .withColumn("fl_populacao_alvo", F.col("faixa_etaria").rlike(REGEX_POP_ALVO))
    .withColumn("quantidade", F.expr("try_cast(quantidade_raw as int)"))
    .withColumn("_bronze_source", F.lit(f"{CATALOG}.{SCHEMA_BRONZE}.resultados_raw"))
    .withColumn("_silver_processed_time", F.current_timestamp())
    .select(
        "codigo_municipio", "municipio", "ano", "grupo_resultado",
        "faixa_etaria_raw", "faixa_etaria", "fl_populacao_alvo",
        "quantidade_raw", "quantidade", "_file_source",
        "_ingestion_time", "_bronze_source", "_silver_processed_time"
    )
    .orderBy("codigo_municipio", "ano", "faixa_etaria")
)

print(f"resultados_citopatologicos: {df_resultados_citopatologicos.count()} linhas preparadas.")
display(df_resultados_citopatologicos.limit(10))

## 5. Validação Estrita e Gravação Final

Validação de **Chaves Primárias (PKs)**, ausência de nulos em colunas obrigatórias e duplicidades **antes** de executar `.write.saveAsTable()`. A gravação só ocorre se todas as validações passarem.

In [0]:
# ---------------------------------------------------------------------------
# 5. VALIDAÇÃO ESTRITA + GRAVAÇÃO FINAL
# ---------------------------------------------------------------------------
# Valida PKs, nulos em colunas obrigatórias e duplicidades ANTES do write.
# Só grava na Silver se todas as validações passarem.

tables_to_validate = [
    ("exames_geral", df_exames_geral,
     ["codigo_municipio", "ano", "mes"],
     ["codigo_municipio", "municipio", "ano", "mes", "dt_competencia"]),
    ("faixa_etaria", df_faixa_etaria,
     ["codigo_municipio", "ano", "faixa_etaria"],
     ["codigo_municipio", "municipio", "ano", "faixa_etaria"]),
    ("adequabilidade", df_adequabilidade,
     ["codigo_municipio", "categoria_adequabilidade", "ano", "mes"],
     ["codigo_municipio", "municipio", "categoria_adequabilidade", "ano", "mes", "dt_competencia"]),
    ("resultados_citopatologicos", df_resultados_citopatologicos,
     ["codigo_municipio", "ano", "faixa_etaria", "grupo_resultado"],
     ["codigo_municipio", "municipio", "ano", "faixa_etaria", "grupo_resultado"]),
]

all_valid = True

print("=" * 70)
print(f"{'TABELA':<30} {'LINHAS':>8} {'DUPS PK':>8} {'NULOS OBRIG':>12} {'STATUS':>8}")
print("=" * 70)

for table_name, df, pk_cols, mandatory_cols in tables_to_validate:
    row_count = df.count()

    # 5.1 Verificar nulos em colunas obrigatórias
    null_exprs = [
        F.count(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), c)).alias(c)
        for c in mandatory_cols
    ]
    null_counts = df.select(null_exprs).collect()[0].asDict()
    total_nulls = sum(null_counts.values())

    # 5.2 Verificar duplicidades na PK
    dup_count = df.groupBy(pk_cols).count().filter("count > 1").count()

    # 5.3 Status
    is_valid = (total_nulls == 0) and (dup_count == 0)
    status = "✅" if is_valid else "❌"
    print(f"{table_name:<30} {row_count:>8} {dup_count:>8} {total_nulls:>12} {status:>8}")

    if total_nulls > 0:
        for col_name, null_cnt in null_counts.items():
            if null_cnt > 0:
                print(f"   ⚠️ {col_name}: {null_cnt} nulos/vazios")
    if dup_count > 0:
        print(f"   ⚠️ {dup_count} linhas duplicadas na PK: {pk_cols}")

    if not is_valid:
        all_valid = False

print("=" * 70)

# 5.4 Gravar na Silver apenas se todas as validações passarem
if all_valid:
    print("\n✅ Todas as validações passaram. Gravando na Silver...")
    for table_name, df, _, _ in tables_to_validate:
        (
            df.write.format("delta")
            .mode("overwrite")
            .option("mergeSchema", "true")
            .saveAsTable(f"{CATALOG}.{SCHEMA_SILVER}.{table_name}")
        )
        print(f"   ✅ {table_name} gravada ({df.count()} linhas)")
    print("\n✅ Camada Silver criada com sucesso!")
else:
    print("\n❌ Validação falhou. Corrija os problemas acima antes de gravar.")

# 5.5 Verificar municípios da 10ª CRS
print("\n📍 Municípios na tabela exames_geral:")
spark.table(f"{CATALOG}.{SCHEMA_SILVER}.exames_geral").select("codigo_municipio", "municipio").distinct().orderBy("codigo_municipio").show(truncate=False)